# The star at every size from $N=3$ to $N=100$

`2026_09_08-check-star-non-monotonues-behavior` simulated the undirected star at **every
integer size** from 3 to 100, $10^6$ repeats each, $r = 1.1$, C++ engine, `max_steps = 10^10`
(so nothing is censored). 98 million runs in total.

The batch exists to settle one question the log-spaced sizes of
`2026_09_02-scaling-more-shapes` could only hint at: $\rho(N)$ for the star is **not
monotone**. It falls from $N=3$, reaches a minimum somewhere in the twenties, and climbs back
towards the $N \to \infty$ limit. Seven sizes cannot tell a real dip from two noisy points;
98 consecutive sizes can.

The notebook does two things:

1. plots what the batch measured, $\rho$ and the conditional fixation time, against $N$;
2. puts the exact analytic star (`analytic_star_fixation_prob` / `analytic_star_fixation_time`,
   the collapsed $2N$-state chain) on top of it, and tests the difference.

The second half is a real test in both directions. The analytics were derived without
reference to this batch, and the batch was run without reference to the analytics, so an
agreement at 98 sizes constrains both: a wrong state collapse and a wrong simulator would have
to be wrong in the same way 98 times.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import binomtest, chi2, kstest, norm

from moran_process.analysis.analysis_utils.io import load_graph_statistics
from moran_process.analysis.analysis_utils.theory import (
    analytic_moran_fc_fixation_prob,
    analytic_moran_fc_fixation_time,
    analytic_star_fixation_prob,
    analytic_star_fixation_time,
)

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

BATCH = "2026_09_08-check-star-non-monotonues-behavior"
BATCH_DIR = Path("simulation_data") / BATCH
R = 1.1
ALPHA = 0.05

C_SIM = "black"
C_EXACT = "#E4C306"   # analytic star
C_BASE = "#9e9e9e"    # complete graph, the amplifier/suppressor baseline
C_RHO = "#1f77b4"
C_TIME = "#d62728"

In [ ]:
# READER, not builder: load_graph_statistics raises if the aggregation job never ran.
star = (
    load_graph_statistics(BATCH_DIR, category_filter="Star", r_filter=R)
    .query("~is_directed")
    .sort_values("n_nodes")
    .reset_index(drop=True)
)

n = star["n_nodes"].to_numpy(float)
r = star["r"].to_numpy(float)

# mean_steps averages over the FIXATING runs only, so its standard error is set by how many
# runs fixated, not by how many were launched. n_grouped counts every run.
star["n_fix"] = (star["prob_fixation"] * star["n_grouped"]).round().astype(int)
star["rho_se"] = np.sqrt(
    star["prob_fixation"] * (1 - star["prob_fixation"]) / star["n_grouped"]
)
star["steps_se"] = star["std_steps"] / np.sqrt(star["n_fix"])

sizes = set(star["n_nodes"])
gapless = sizes == set(range(int(n.min()), int(n.max()) + 1))
print(
    f"{BATCH}\n"
    f"  r = {sorted(star['r'].unique())}, N = {int(n.min())}..{int(n.max())} "
    f"({len(star)} sizes, no gaps: {gapless})\n"
    f"  {star['n_grouped'].iloc[0]:,} repeats per size, {star['n_grouped'].sum():,} runs, "
    f"{star['n_censored'].sum()} censored"
)

dip = star.loc[star["prob_fixation"].idxmin()]
print(f"  minimum simulated rho = {dip['prob_fixation']:.6f} at N = {int(dip['n_nodes'])}")

star[["graph_name", "n_nodes", "prob_fixation", "rho_se",
      "mean_steps", "steps_se", "std_steps", "n_fix"]].head(8)

## What the batch measured

Three panels, all against $N$:

- **left**, $\rho$ over the whole range. The fall from $N=3$ dominates the axis, which is why
  the middle panel exists.
- **middle**, the same $\rho$ from $N=10$ up. This is the panel the batch was run for: the
  curve has a genuine minimum and then rises, and the error bars are small enough
  ($\pm 0.0007$ at 95%) that the turn is not noise.
- **right**, the conditional fixation time, log-log. A power law fitted above $N=30$ gives the
  exponent in the legend; the complete graph of the same size is drawn for scale.

The complete-graph curves are the baseline that defines amplification: above it in $\rho$ is
an amplifier, below is a suppressor. At $r = 1.1$ the complete graph sits near $\rho \approx
0.09$, so the star is a strong amplifier at every size here, dip or no dip.

In [ ]:
star["n_censored"].unique()

In [ ]:
from moran_process.analysis.analysis_utils.plotly_prototype import (
    plot_outcome_vs_property_plotly,
    plot_two_property_effect_plotly,
)
plot_outcome_vs_property_plotly(
    star,
    "n_nodes",
    "prob_fixation",
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))
grid = np.arange(3, 101)

def draw_rho(ax, frame, title, baseline=True):
    if baseline:
        ax.plot(grid, analytic_moran_fc_fixation_prob(grid, R), "--", color=C_BASE, lw=1.8,
                label="complete graph (baseline)", zorder=1)
    ax.errorbar(frame["n_nodes"], frame["prob_fixation"], yerr=1.96 * frame["rho_se"],
                fmt="o", ms=4, color=C_SIM, capsize=2, lw=1, zorder=3,
                label=r"star, simulated ($10^6$ runs, 95% CI)")
    ax.set(xlabel="N", ylabel=r"$\rho$", title=title)

draw_rho(axes[0], star, rf"Fixation probability, $r={R}$")
axes[0].legend(fontsize=8.5)

zoom = star[star["n_nodes"] >= 10]
# No baseline in the zoom: the complete graph sits far below this window.
draw_rho(axes[1], zoom, "The same, from N = 10 (the dip)", baseline=False)
axes[1].set_ylim(zoom["prob_fixation"].min() - 0.0015, zoom["prob_fixation"].max() + 0.0015)
axes[1].axvline(dip["n_nodes"], color=C_RHO, lw=1.2, ls=":", zorder=1)
axes[1].annotate(f"minimum at N = {int(dip['n_nodes'])}",
                 xy=(dip["n_nodes"], dip["prob_fixation"]),
                 xytext=(dip["n_nodes"] + 12, dip["prob_fixation"] - 0.0008),
                 fontsize=9, color=C_RHO,
                 arrowprops=dict(arrowstyle="->", color=C_RHO, lw=1.2))
ax = axes[2]
tail = star[star["n_nodes"] >= 30]
slope, intercept = np.polyfit(np.log(tail["n_nodes"]), np.log(tail["mean_steps"]), 1)
ax.plot(grid, analytic_moran_fc_fixation_time(grid, R), "--", color=C_BASE, lw=1.8,
        label="complete graph (baseline)")
ax.plot(star["n_nodes"], star["mean_steps"], "o", ms=4, color=C_SIM, label="star, simulated")
fit_grid = grid[grid >= 30]  # drawn only over the range it was fitted on
ax.plot(fit_grid, np.exp(intercept) * fit_grid**slope, "-", color=C_TIME, lw=1.5, alpha=0.8,
        label=rf"fit above N=30: $\propto N^{{{slope:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="N", ylabel="mean steps to fixation",
       title="Conditional fixation time")
ax.legend(fontsize=8.5)

fig.suptitle(f"{BATCH}: undirected star, every N from 3 to 100", y=1.03)
fig.tight_layout()
plt.show()

## The analytic solution against the simulation

`analytic_star_fixation_prob` and `analytic_star_fixation_time` collapse the $2^N$-state Moran
chain to the $2N$ pairs $(\text{hub type}, \text{mutant leaf count})$ and solve two banded
systems. Both quantities are conditioned and counted exactly the way the simulation reports
them: elementary birth-death events including null events, mutant placed uniformly (the hub
with probability $1/N$), time averaged over fixating runs only.

Three statistics per metric, because they fail differently:

| statistic | what it catches |
|---|---|
| per-size $z$ and $p$ | a single size where the two disagree |
| Fisher-combined $p$ over the 98 sizes | a systematic bias too small to see at any one size |
| KS of the 98 $z$'s against $N(0,1)$ | residuals that are the wrong *shape*, e.g. an SE that is itself wrong |

Bonferroni is reported alongside Fisher because they answer different questions: Bonferroni
asks whether **any** size disagrees, Fisher whether the sizes disagree **together**. With 98
independent tests, one $p < 0.05/98$ is a 1-in-20 event even when everything is correct, so a
marginal Bonferroni with a clean KS is the expected tail, not a defect.

In [ ]:
star["rho_exact"] = analytic_star_fixation_prob(n, r)
star["steps_exact"] = analytic_star_fixation_time(n, r)

star["rho_z"] = (star["prob_fixation"] - star["rho_exact"]) / np.sqrt(
    star["rho_exact"] * (1 - star["rho_exact"]) / star["n_grouped"]
)
star["rho_p"] = [
    binomtest(int(k), int(m), p).pvalue
    for k, m, p in zip(star["n_fix"], star["n_grouped"], star["rho_exact"])
]
star["rho_rel"] = star["prob_fixation"] / star["rho_exact"] - 1

# The exact SE of mean_steps would need the exact second moment; std_steps is the measured
# one over 1.6e5+ fixating runs, which is the same quantity to four digits.
star["steps_z"] = (star["mean_steps"] - star["steps_exact"]) / star["steps_se"]
star["steps_p"] = 2 * norm.sf(star["steps_z"].abs())
star["steps_rel"] = star["mean_steps"] / star["steps_exact"] - 1

for label, zcol, pcol, rcol in (("rho ", "rho_z", "rho_p", "rho_rel"),
                                ("time", "steps_z", "steps_p", "steps_rel")):
    z, p = star[zcol], star[pcol]
    fisher = chi2.sf(-2 * np.log(p).sum(), 2 * len(p))
    bonf = min(1.0, p.min() * len(p))
    ks = kstest(z, "norm").pvalue
    worst = star.loc[z.abs().idxmax()]
    print(
        f"{label}  mean z {z.mean():+.3f}  sd z {z.std(ddof=1):.3f}  "
        f"|z| max {z.abs().max():.2f} (at N = {int(worst['n_nodes'])})  "
        f"|rel| max {star[rcol].abs().max():.2e}\n"
        f"      Fisher p = {fisher:.3f}   Bonferroni min p x {len(p)} = {bonf:.3f}   "
        f"KS(z, N(0,1)) p = {ks:.3f}   |z| < 1.96 in {(z.abs() < 1.96).mean():.1%} of sizes"
    )

# The dip is a property of the exact solution, not of Monte Carlo noise: the analytic curve
# has its minimum at the same N the simulation does.
rho_grid = analytic_star_fixation_prob(np.arange(3, 101), R)
print(f"\nanalytic argmin N = {np.arange(3, 101)[rho_grid.argmin()]}, "
      f"simulated argmin N = {int(dip['n_nodes'])}")

cols = ["n_nodes", "prob_fixation", "rho_exact", "rho_z", "rho_p",
        "mean_steps", "steps_exact", "steps_rel", "steps_z", "steps_p"]
display(star.loc[star["rho_z"].abs().nlargest(6).index, cols].style.format({
    "prob_fixation": "{:.6f}", "rho_exact": "{:.6f}", "rho_z": "{:+.2f}", "rho_p": "{:.4f}",
    "mean_steps": "{:.5g}", "steps_exact": "{:.5g}", "steps_rel": "{:+.4%}",
    "steps_z": "{:+.2f}", "steps_p": "{:.3f}",
}).hide(axis="index").set_caption("the 6 sizes where rho deviates most"))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5))
grid = np.arange(3, 101)

ax = axes[0, 0]
ax.plot(grid, analytic_star_fixation_prob(grid, R), "-", color=C_EXACT, lw=2.5,
        label="analytic, exact at every N", zorder=2)
ax.errorbar(zoom["n_nodes"], zoom["prob_fixation"], yerr=1.96 * zoom["rho_se"],
            fmt="o", ms=4, color=C_SIM, capsize=2, lw=1, zorder=3,
            label=r"simulated, $10^6$ runs (95% CI)")
ax.set(xlim=(9, 101),
       ylim=(zoom["prob_fixation"].min() - 0.0015, zoom["prob_fixation"].max() + 0.0015),
       xlabel="N", ylabel=r"$\rho$", title=r"Fixation probability (from N = 10)")
ax.legend(fontsize=9)

ax = axes[0, 1]
ax.plot(grid, analytic_star_fixation_time(grid, R), "-", color=C_EXACT, lw=2.5,
        label="analytic, exact", zorder=2)
ax.plot(star["n_nodes"], star["mean_steps"], "o", ms=4, color=C_SIM, zorder=3,
        label="simulated")
ax.set(xscale="log", yscale="log", xlabel="N", ylabel="mean steps to fixation",
       title="Conditional fixation time")
ax.legend(fontsize=9)

ax = axes[1, 0]
ax.axhspan(-1.96, 1.96, color="#4CAF50", alpha=0.13, lw=0,
           label="95% of sizes land here if exact")
ax.axhline(0, color="black", lw=1)
ax.plot(star["n_nodes"], star["rho_z"], "o-", ms=3.5, lw=0.9, color=C_RHO, label=r"$\rho$")
ax.plot(star["n_nodes"], star["steps_z"], "s-", ms=3.5, lw=0.9, color=C_TIME,
        label="mean time")
ax.set(xlabel="N", ylabel="z  (simulated minus exact, in SEs)",
       title="Standardized deviation at each size", ylim=(-4.2, 5.4))
ax.legend(fontsize=9, loc="upper right", ncol=3)

ax = axes[1, 1]
ax.axhline(0, color="black", lw=1)
ax.plot(star["n_nodes"], 100 * star["rho_rel"], "o-", ms=3.5, lw=0.9, color=C_RHO,
        label=r"$\rho$")
ax.plot(star["n_nodes"], 100 * star["steps_rel"], "s-", ms=3.5, lw=0.9, color=C_TIME,
        label="mean time")
ax.set(xlabel="N", ylabel="simulated / exact  minus 1  (%)",
       title="Relative agreement (note the y scale)")
ax.legend(fontsize=9)

fig.suptitle("Exact star vs 98,000,000 simulated runs, N = 3..100, r = 1.1", y=1.0)
fig.tight_layout()
plt.show()

## What this settles

- The **dip is real and it is analytic**. The exact solution has its minimum at the same $N$
  the simulation does, so it is a property of the finite-$N$ star chain rather than of the
  sampling. The star is an amplifier at every size here; what is non-monotone is *how much*.
- **The analytic star reproduces the simulator at 98 consecutive sizes**: the largest relative
  disagreement anywhere is 0.8% on $\rho$ and 0.3% on the mean time, both at the size of their
  own Monte Carlo error, and the 98 residuals are indistinguishable from $N(0,1)$ (KS $p =
  0.90$ and $0.96$). Either function may now be used as a reference curve without re-simulating.
- The one size Bonferroni flags is $N = 32$ ($|z| = 3.6$, min $p \times 98 = 0.03$). With
  98 independent tests that is the expected tail rather than a defect, and the KS and Fisher
  statistics both say the residuals as a whole are clean. A re-run at a different `batch_seed`
  would settle it for the cost of one size.
- Both are therefore cross-validated: the derivation is not checked by the code that uses it,
  and the engine is not checked by the graph it runs on, but 98 independent agreements at
  $10^6$ runs each is not something two independent errors produce.

The previous version of this notebook ran five other checks against
`2026_09_02-scaling-more-shapes` (brute force over all $2^N$ states, the second moment, an
exact-PMF KS test, the $N \to \infty$ asymptotic, and the directed star as a $\rho = 1/N$
control). Those tests are orthogonal to the ones here and can be folded back in; this rewrite
is deliberately about the dense-$N$ batch and the two curves it was run to produce.